# Week 7 실습

> "외부 업체가 AUC 0.9짜리 가입 예측 모델을 만들어 왔어요. 바로 도입해도 될까요?" — 마케팅팀장

오늘 실습은 이 모델을 바로 써도 되는지 확인하는 것이다. 어떤 변수를 넣고, 데이터를 어떻게 나누고, 어느 구간에서 전처리를 학습하느냐에 따라 점수의 의미가 달라진다. 높은 점수가 실제로 믿을 만한 점수인지 확인하는 것이 목표.

| Part | 내용 | 셀 |
|---|---|---|
| 1 | 데이터와 예측 시점 확인 | 실행 |
| 2 | 통화시간을 넣으면 점수가 올라간다 | 예측, 빈칸, 해석 |
| 3 | 무작위로 나눌까, 시간순으로 나눌까 | 예측, 실행, 해석 |
| 4 | 섞지 않으면 시간순 검증일까 | 예측, 실행, 해석 |
| 5 | 학습 점수만 높으면 좋은 모델일까 | 해석 |
| 6 | 코드 점검, 팀장에게 보낼 결론 정리 | 해석 |

셀 표시
- [실행] 실행만 하는 셀
- [빈칸] `TODO`를 채우는 셀
- [예측] 코드를 돌리기 전에 예상을 적는 칸. 틀린 예측도 기록해 둔다.
- [해석] 결과를 보고 해석을 적는 칸



## 기본 세팅 [실행]

공용 함수가 구현된 저장소에서는 공용 함수를 사용한다. 노트북 파일만 따로 열거나 공용 함수가 아직 미구현이면 아래 셀에 들어 있는 같은 함수를 사용한다. 데이터 파일이 없으면 공식 UCI 파일을 한 번 내려받음.

원본 행 순서는 그대로 두고, 정답인 `y`는 예측에 쓰는 열에서 제외한다.


In [ ]:
# 확인할 것: OK (45211, 18), pandas·sklearn 버전, 한글 폰트 안내.
import sys
from pathlib import Path
# 현재 폴더·상위 폴더·바로 아래의 저장소 폴더를 확인한다.
_candidates = [Path.cwd(), *Path.cwd().parents,
               Path.cwd() / "DAstudy", Path.cwd() / "DAstudy-week07"]
ROOT = next((p for p in _candidates if (p / "common/load_data.py").is_file()), None)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from matplotlib import font_manager
from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, TimeSeriesSplit, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
# 공용 함수가 구현된 저장소에서는 공용 함수를 쓰고, 미구현이면 노트북의 구현을 쓴다.
_USE_COMMON = False
if ROOT is not None:
    sys.path.insert(0, str(ROOT))
    try:
        from inspect import getsource
        from common.load_data import load_bank
        from common.bank.preprocess import add_prev_contact_flag
        from common.evaluate import make_pipeline, time_split
        _USE_COMMON = all("raise NotImplementedError" not in getsource(fn)
                          for fn in [add_prev_contact_flag, make_pipeline, time_split])
    except (ImportError, OSError, TypeError):
        _USE_COMMON = False
if _USE_COMMON:
    DATA_DIR = ROOT / "data"
    print("공용 함수 사용:", ROOT)
else:
    DATA_DIR = (ROOT if ROOT is not None else Path.cwd()) / "data"
    print("공용 함수가 없거나 미구현: 노트북 안의 동일한 함수 사용")
    def add_prev_contact_flag(df):
        """week04 / D-004 [bank]: pdays == -1 → 이전 연락 여부 플래그(prev_contact) 생성.

        -1은 "이전 연락 없음"을 뜻하는 코드값이라 경과 일수와 분리해야 한다.
        pdays의 -1을 NaN으로 바꿀지 그대로 둘지는 week04 모임에서 결정.
        week07 모델링에서 이 함수를 적용한 데이터를 쓴다 (unknown은 하나의 범주로 유지).
        """
        result = df.copy()
        # 초안 정책: 미접촉 코드값은 NaN으로 분리하고, 수치 대치는 train 안에서만.
        result["prev_contact"] = result["pdays"].ne(-1).astype("Int8").mask(result["pdays"].isna())
        result["pdays"] = result["pdays"].mask(result["pdays"].eq(-1))
        return result

    def make_pipeline(num_cols, cat_cols, model=None):
        """week07 / D-007: ColumnTransformer(스케일러 + 원핫) + 모델을 하나의 Pipeline으로.

        - 전처리는 반드시 Pipeline 안에서 train에만 fit
        - OneHotEncoder(handle_unknown="ignore") — 시간순 분할에서 test에만 나오는 범주 대비
        """
        from sklearn.compose import ColumnTransformer
        from sklearn.impute import SimpleImputer
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import OneHotEncoder, StandardScaler

        num_cols, cat_cols = list(num_cols), list(cat_cols)
        if set(num_cols) & set(cat_cols):
            raise ValueError("수치형·범주형 열은 중복될 수 없습니다.")
        if not num_cols and not cat_cols:
            raise ValueError("모델에 사용할 열이 필요합니다.")
        preprocess = ColumnTransformer([
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                              ("scaler", StandardScaler())]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ], remainder="drop")
        estimator = LogisticRegression(max_iter=3000, solver="lbfgs", random_state=42) if model is None else model
        return Pipeline([("preprocess", preprocess), ("model", estimator)])

    def time_split(df, train_frac=0.8):
        """week07 / D-007: 행 순서(= 시간순) 기준 앞 train_frac은 train, 나머지는 test.

        Bank 전용 (Home Credit은 신청 날짜가 없어 시간순 분할 불가).
        """
        import math
        from numbers import Real

        if isinstance(train_frac, bool) or not isinstance(train_frac, Real) or not math.isfinite(train_frac) or not 0 < train_frac < 1:
            raise ValueError("train_frac은 0과 1 사이의 유한한 실수여야 합니다.")
        cut = int(len(df) * train_frac)
        if cut == 0 or cut == len(df):
            raise ValueError("train과 test에 각각 최소 1행이 필요합니다.")
        return df.iloc[:cut].copy(), df.iloc[cut:].copy()

    def load_bank(check_shape=True):
        result = pd.read_csv(DATA_DIR / "bank-full.csv", sep=";")
        if check_shape and result.shape != (45211, 17):
            raise ValueError(f"bank-full.csv의 크기가 {result.shape}입니다. (45211, 17) 버전이 필요합니다.")
        return result

# 데이터가 없을 때만 공식 공개 파일을 받는다. 기존 파일은 덮어쓰지 않는다.
_data_path = DATA_DIR / "bank-full.csv"
if not _data_path.exists():
    from io import BytesIO
    from zipfile import ZipFile
    from urllib.request import urlopen
    _uploaded = Path.cwd() / "bank-full.csv"
    if _uploaded.exists():
        _csv_bytes = _uploaded.read_bytes()
    else:
        print("Bank Marketing 데이터를 UCI에서 내려받는 중...")
        try:
            with urlopen("https://archive.ics.uci.edu/static/public/222/bank+marketing.zip", timeout=30) as response:
                _zip_bytes = response.read()
            with ZipFile(BytesIO(_zip_bytes)) as outer:
                with ZipFile(BytesIO(outer.read("bank.zip"))) as inner:
                    _csv_bytes = inner.read("bank-full.csv")
        except Exception as exc:
            raise RuntimeError(
                "UCI 데이터 다운로드에 실패했습니다. bank-full.csv를 현재 작업 폴더에 업로드한 뒤 "
                "이 셀을 다시 실행하세요. 현재 폴더: " + str(Path.cwd())
            ) from exc
    # 다른 버전이나 불완전한 파일을 캐시에 저장하지 않도록 확인한다.
    from io import BytesIO
    _check = pd.read_csv(BytesIO(_csv_bytes), sep=";")
    if _check.shape != (45211, 17):
        raise ValueError("bank-full.csv (45211, 17) 파일이 필요합니다. bank-additional-full.csv와 구분하세요.")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _data_path.write_bytes(_csv_bytes)

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 100
_installed = {f.name for f in font_manager.fontManager.ttflist}
for _font in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Nanum Gothic", "Noto Sans CJK KR", "Noto Sans KR"]:
    if _font in _installed:
        plt.rcParams["font.family"] = _font
        break
else:
    print("한글 폰트가 없으면 나눔고딕 설치 후 커널을 재시작하세요.")
plt.rcParams["axes.unicode_minus"] = False
raw = load_bank()
df = add_prev_contact_flag(raw)
y = df["y"].map({"no": 0, "yes": 1})
assert y.notna().all()
print("OK", df.shape)
print("pandas", pd.__version__, "sklearn", sklearn.__version__)


---
## Part 1. 데이터와 예측 시점 확인 [실행]

모델을 만들기 전에 데이터에 무엇이 들어 있는지부터 확인한다. 이 Part는 실행만!

### 1-1. 어떤 데이터인가?

Bank Marketing은 포르투갈 한 은행에서 **전화로 정기예금 가입을 권유한 기록**이다. 정기예금은 일정 기간 돈을 맡기고 이자를 받는 상품이다. 고객의 나이·직업·대출 여부 같은 정보와 연락 기록, 최종 가입 여부가 들어 있음.

| 항목 | 내용 |
|---|---|
| 출처 | [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) |
| 사용하는 파일 | `bank-full.csv` — 45,211행 × 17열, 구분자는 세미콜론(`;`) |
| 기간·순서 | 2008년 5월부터 2010년 11월까지의 기록. 행이 날짜순으로 정렬되어 있고 연도 열은 없음 |
| 한 행의 의미 | 캠페인 대상 기록 한 건. 고객 ID가 없어 서로 다른 고객 45,211명이라고 볼 수는 없음 |
| 예측할 정답 | `y`: 정기예금 가입 여부 (`yes` / `no`) |
| 가입 비율 | `yes` 5,289건, 약 11.7%. 나머지는 `no` |
| 결과가 설명하는 대상 | 은행 전체 고객이 아니라, 이 기간에 마케팅 대상으로 선정돼 연락받은 고객의 기록 |

### 1-2. 어떤 변수가 들어 있는가?

| 변수 | 뜻 | 오늘 확인할 점 |
|---|---|---|
| `age`, `job`, `marital`, `education` | 나이, 직업, 혼인 상태, 학력 | 고객의 기본 정보 |
| `default`, `balance`, `housing`, `loan` | 신용 불량 여부, 연평균 잔액(유로), 주택대출·개인대출 여부 | 잔액에는 음수가 들어갈 수 있음 |
| `contact`, `day`, `month` | 마지막 연락 수단과 날짜의 일·월 | 이번 초안에서는 사전 가용성이 불확실해 비교 모델 모두에서 제외 |
| `duration` | 마지막 통화 시간(초) | **통화가 끝나야 알 수 있음.** Part 2에서 포함/제외 비교 |
| `campaign` | 이번 캠페인 연락 횟수, 마지막 연락 포함 | 첫 통화 전에는 최종 횟수를 알 수 없어 이번 모델에서 제외 |
| `pdays` | 이전 캠페인에서 마지막으로 연락한 뒤 지난 일수 | `-1`은 '이전에 연락한 적 없음'이라는 코드 |
| `previous`, `poutcome` | 이전 캠페인 연락 횟수와 결과 | 이번 통화 전의 연락 이력 |
| `y` | 가입 여부 | 맞혀야 할 정답. 예측 입력에는 넣지 않음 |

`unknown`은 값이 알려지지 않았다는 뜻의 문자열이다. 오늘은 하나의 범주로 유지한다.

### 1-3. 이 데이터로 무엇을 할까?

은행이 **전화를 걸기 전에 누구에게 걸지 고르는 상황**을 생각해보자. 과거 기록에서 고객 정보와 가입 여부의 관계를 학습하고, 학습에 쓰지 않은 기록에서도 가입 가능성을 잘 구분하는지 확인한다.

- 예측에 넣는 정보(`X`): 고객 정보와 이전 연락 이력 등
- 맞히려는 정답(`y`): 가입이면 1, 미가입이면 0으로 바꿔 사용
- 오늘 비교할 것: 통화시간을 넣었을 때와 뺐을 때, 기록을 섞었을 때와 시간순으로 나눴을 때의 평가 결과

이 데이터는 **통화 후에야 아는 값(`duration`)**과 **기록의 시간 순서**가 있어 누수와 검증 방식을 같이 확인하기 좋다. 왜 점수가 달라지는지는 Part 2부터 직접 본다.

### 1-4. 먼저 적용할 처리

`pdays = -1`은 '-1일'이 아니다. 이번 초안에서는 이전 연락 여부(`prev_contact`)를 따로 만들고, -1은 `NaN`으로 바꾼다. 이 열이 하나 추가돼 세팅 셀에서는 `OK (45211, 18)`이 나옴.

비어 있는 수치를 채울 때 쓰는 중앙값은 학습 데이터에서만 계산한다. 이 과정과 스케일링, 원-핫 인코딩을 `Pipeline`으로 묶어 둠.


In [ ]:
# 확인할 것: 가입률, 이전 연락 플래그, 원본 pdays가 바뀌지 않았는지.
display(pd.DataFrame({"건수": y.value_counts(), "비율": y.value_counts(normalize=True)}))
display(pd.crosstab(raw["pdays"].eq(-1), df["prev_contact"]))
assert df["pdays"].isna().equals(raw["pdays"].eq(-1))
assert "prev_contact" not in raw.columns
# 첫 통화 전 기준에서 근거가 애매한 최종 기록은 보수적으로 제외한다.
# contact도 실제 연락 수단이므로 사전 채널이 확정됐다는 근거 없이는 제외.
AMBIGUOUS = ["campaign", "day", "month", "contact"]
X_with = df.drop(columns=["y", *AMBIGUOUS])
X = X_with.drop(columns="duration")
assert "y" not in X and "sub" not in X

def pipeline_for(frame, model=None):
    nums = frame.select_dtypes(include="number").columns.tolist()
    cats = frame.columns.difference(nums, sort=False).tolist()
    return make_pipeline(nums, cats, model)

def evaluate(frame, train_idx, test_idx, model=None):
    pipe = pipeline_for(frame, model)
    pipe.fit(frame.iloc[train_idx], y.iloc[train_idx])
    return pipe, {
        "train AUC": roc_auc_score(y.iloc[train_idx], pipe.predict_proba(frame.iloc[train_idx])[:, 1]),
        "test AUC": roc_auc_score(y.iloc[test_idx], pipe.predict_proba(frame.iloc[test_idx])[:, 1]),
        "train 가입률": y.iloc[train_idx].mean(), "test 가입률": y.iloc[test_idx].mean(),
        "train 행": len(train_idx), "test 행": len(test_idx),
    }
# 분할을 한 번 만들고 모든 비교에 재사용한다.
random_train, random_test = train_test_split(np.arange(len(df)), test_size=0.2,
                                             stratify=y, random_state=42)
train_df, test_df = time_split(df)
cut = len(train_df)
time_train, time_test = np.arange(cut), np.arange(cut, len(df))
# 실제 배포 검증이라면 뒤 20%를 봉인하고, 아래 CV는 앞 80%에서만 진행한다.
assert time_train.max() < time_test.min()
print("누수 제외 입력:", X.columns.tolist())

---
## Part 2. 통화시간을 넣으면 점수가 올라간다 [예측, 빈칸, 해석]

`duration`은 이번 통화가 몇 초 동안 이어졌는지다. 통화가 끝나야 알 수 있는 값인데, 가입 여부와는 관련이 클 수 있다.

같은 학습·평가 데이터에서 `duration`만 넣었다 뺐다 해 본다. 다른 조건은 그대로 두고, `campaign`, `day`, `month`, `contact`는 두 모델 모두에서 제외했음.

예측: `duration`을 넣은 모델의 AUC가 더 높을까? → (    )

질문: 점수가 더 높다면, 전화를 걸 사람을 고르는 데 이 모델을 써도 될까? 결과를 보고 다시 생각해보자.


In [ ]:
# 확인할 것: 같은 무작위 분할에서 duration 포함/제외의 test AUC 차이.
_, leak_row = evaluate(TODO, random_train, random_test)
random_pipe, clean_row = evaluate(TODO, random_train, random_test)
leak_table = pd.DataFrame([leak_row, clean_row], index=["duration 포함", "duration 제외"])
display(leak_table)
leak_table[["train AUC", "test AUC"]].rename(columns={"train AUC": "학습 AUC", "test AUC": "평가 AUC"}).plot.bar(ylim=(0, 1), rot=0, title="통화시간 포함 여부와 AUC")
plt.ylabel("ROC-AUC")
plt.tight_layout()
plt.show()

[해석] `duration`을 넣었을 때와 뺐을 때 AUC가 얼마나 달라졌는가? 전화 전에 쓸 모델이라면 어느 쪽을 사용해야 하는가? (숫자와 이유를 한 줄로)

-


---
## Part 3. 무작위로 나눌까, 시간순으로 나눌까 [예측, 실행, 해석]

이번에는 `duration`을 뺀 상태에서 데이터를 나누는 방법만 바꿔 본다.

- 무작위 분할: 전체 기록을 섞어서 80%는 학습, 20%는 평가에 사용
- 시간순 분할: 앞 80%로 학습하고 뒤 20%로 평가. 원본 행 순서는 그대로 둔다.

학습 데이터는 모델이 규칙을 배우는 데 쓰는 기록이고, 평가 데이터는 배우지 않은 기록에서도 잘 맞히는지 확인하는 데 쓰는 기록이다.

예측: 시간순으로 나누면 AUC는 반드시 낮아질까? → (    )

예측: 두 구간의 가입률은 비슷할까, 다를까? → (    )

체크: 뒤 20%가 정확히 '내년'을 뜻하는 것은 아니다. 고객 ID가 없어 같은 고객이 양쪽에 들어갔는지도 확인할 수 없음. 결과표에서 AUC와 가입률을 같이 본다.


In [ ]:
# 확인할 것: AUC뿐 아니라 각 구간의 행 수·가입률이 얼마나 다른지.
time_pipe, time_row = evaluate(X, time_train, time_test)
split_table = pd.DataFrame([clean_row, time_row], index=["무작위", "시간순"])
display(split_table)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
split_table[["train AUC", "test AUC"]].rename(columns={"train AUC": "학습 AUC", "test AUC": "평가 AUC"}).plot.bar(ax=axes[0], ylim=(0, 1), rot=0, title="분할별 AUC")
split_table[["train 가입률", "test 가입률"]].rename(columns={"train 가입률": "학습 가입률", "test 가입률": "평가 가입률"}).plot.bar(ax=axes[1], rot=0, title="분할별 가입률")
axes[0].set_ylabel("ROC-AUC")
axes[1].set_ylabel("가입 비율")
plt.tight_layout()
plt.show()

[해석] 다음 캠페인에 쓸 모델이라면 어떤 분할이 사용 상황과 가까운가? AUC가 높은 쪽을 고르는 것만으로 충분한가? (가입률도 근거로)

-


---
## Part 4. 섞지 않으면 시간순 검증일까 [예측, 실행, 해석]

한 번 나눈 결과만 보는 대신, 학습·평가 구간을 바꿔 여러 번 확인하는 것이 교차검증이다. 아래 세 방법을 비교한다.

| 방법 | 나누는 방식 |
|---|---|
| KFold 섞음 | 기록을 섞어 5개 구간으로 나누고 하나씩 평가 |
| KFold 순서 | 순서대로 5개 구간으로 나누고 하나씩 평가 |
| TimeSeriesSplit | 과거 구간으로 학습하고 그다음 구간으로 평가 |

예측: `KFold(shuffle=False)`면 항상 과거 기록으로 미래를 예측하는 것일까? → (    )

이 Part는 발제자가 미리 실행한 결과를 같이 본다. **KFold 순서의 fold 1과 시계열의 fold 1**에서 학습 마지막 행과 평가 첫 행을 비교하기.

교차검증은 앞 80% 안에서만 한다. 뒤 20%는 교차검증 학습에 넣지 않았음.


In [ ]:
# 확인할 것: fold별 AUC·가입률과 미래 학습 행 비율. TimeSeriesSplit은 미래 학습 비율이 0.
X_dev, y_dev = X.iloc[:cut], y.iloc[:cut]
splitters = {
    "KFold 섞음": KFold(n_splits=5, shuffle=True, random_state=42),
    "KFold 순서": KFold(n_splits=5, shuffle=False),
    "시계열": TimeSeriesSplit(n_splits=5),
}
cv_rows = []
for name, splitter in splitters.items():
    folds = list(splitter.split(X_dev, y_dev))
    scores = cross_val_score(pipeline_for(X_dev), X_dev, y_dev,
                             cv=folds, scoring="roc_auc", error_score="raise", n_jobs=1)
    for fold, ((tr, va), auc) in enumerate(zip(folds, scores), 1):
        cv_rows.append({"방식": name, "fold": fold, "AUC": auc,
                        "train 가입률": y_dev.iloc[tr].mean(), "valid 가입률": y_dev.iloc[va].mean(),
                        "train 행": len(tr), "valid 행": len(va),
                        "train 최대 행": tr.max(), "valid 최소 행": va.min(),
                        "valid 시작 이후 train 비율": (tr > va.min()).mean()})
cv_table = pd.DataFrame(cv_rows)
display(cv_table.loc[cv_table["fold"].eq(1),
    ["방식", "fold", "AUC", "train 최대 행", "valid 최소 행", "valid 시작 이후 train 비율"]])
display(cv_table.groupby("방식")["AUC"].agg(["mean", "std"]))
# 모임 후: cv_table 전체를 열어 15개 fold의 가입률과 점수를 확인한다.


[해석] KFold 순서 방식과 TimeSeriesSplit에서 학습에 들어가는 행은 어떻게 다른가? 미래 기록이 학습에 들어가는 경우를 결과표에서 찾는다.

-


---
## Part 5. 학습 점수만 높으면 좋은 모델일까 [해석]

같은 시간순 분할에서 로지스틱 회귀와 Random Forest를 비교한 결과다. `duration`은 두 모델 모두에서 제외했고, `random_state=42`로 맞췄음.

아래 숫자는 발제자가 전체 해설본을 미리 실행해 둔 값이다. 여기서 새로 학습하지는 않음.

| 모델 | 학습 AUC | 평가 AUC |
|---|---:|---:|
| 로지스틱 회귀 | 0.6193 | 0.7174 |
| Random Forest | 0.9996 | 0.6496 |

[해석] "학습 AUC가 거의 1이니까 Random Forest가 더 좋은 모델이다"라고 해도 되는가? 학습·평가 점수를 함께 보고 이유를 적는다. (한 줄)

-


---
## Part 6. 코드 점검, 팀장에게 보낼 결론 정리 [해석]

### 6-1. 이 코드로 나온 점수, 믿어도 될까?

아래는 점검 연습용으로 만든 코드다. 지금까지 본 내용을 바탕으로 문제를 찾아본다.

```python
X = df.drop(columns="y")
Z = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(Z, y, random_state=42)
```

| 점검할 것 | 문제와 근거 |
|---|---|
| 예측 시점에 알 수 없는 열이 들어갔는가 |  |
| 전처리를 학습 데이터에서만 배웠는가 |  |
| 나누는 방식이 실제 사용 상황과 맞는가 |  |

### 6-2. 팀장에게 어떻게 답할까?

① "바로 도입 / 추가 검증 후 판단" 중 하나를 고르고, 오늘 본 숫자 하나를 근거로 적는다.

-

② 외부 업체에 꼭 물어볼 질문 세 가지를 정한다. 입력 변수와 예측 시점, 학습·평가 기간, 전처리 범위를 생각해보기.

1.
2.
3.

### 마무리 [해석]

오늘의 결론 세 줄. 점수의 의미를 바꾼 선택을 적는다.

1. 어떤 변수를 넣느냐 →
2. 데이터를 어떻게 나누느냐 →
3. 학습·평가 중 어느 점수를 보느냐 →


> 미래 캠페인을 검증할 때는 원본 순서를 유지한 시간순 분할과 학습 구간 안의 시계열 교차검증을 사용한다. 전처리는 Pipeline 안에서 학습 구간에만 fit한다. 예측 시점에 알 수 없는 변수는 제외한다.

제출: 내 폴더의 `worksheet.md`, `analysis.ipynb`와 `summary.md` 한 줄. **11/11 (수) 23:59까지**.

모임 후 선택: 합성 Hotel 도입, 전체 데이터에 전처리를 fit한 버전 비교, Random Forest 직접 학습, 전체 fold 결과 확인, 새 AI 답변과 비교. 오늘 본 뒤 20% 점수로 모델을 고르거나 튜닝하지 않는다. 이후 최종 성능을 확인하려면 새로운 독립 평가가 필요함.

참고: [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing), [교차검증](https://scikit-learn.org/stable/modules/cross_validation.html), [데이터 누수와 Pipeline](https://scikit-learn.org/stable/common_pitfalls.html)
